# Energy Forecast PT - Model Training (No Lags)

Treinamento de modelo **sem features de lag** para permitir previsões sem histórico.

**Trade-off:**
- ✅ Funciona sem dados históricos (apenas weather + temporal)
- ⚠️ Menor precisão que o modelo completo

**Modelos:**
- Random Forest
- XGBoost
- LightGBM
- CatBoost

## 1. Setup

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
import json
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

from src.models.evaluation import ModelEvaluator

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print("Bibliotecas carregadas")
print(f"Inicio: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Bibliotecas carregadas
Inicio: 2025-10-07 10:36:31


## 2. Carregar Dados

In [2]:
# Carregar dados processados
df = pd.read_parquet('../data/processed/processed_data.parquet')
print(f"Dados carregados: {len(df):,} linhas")
print(f"Periodo: {df['timestamp'].min()} a {df['timestamp'].max()}")
print(f"Regioes: {df['region'].unique()}")
print(f"Consumo - Min: {df['consumption_mw'].min():.2f} MW, Max: {df['consumption_mw'].max():.2f} MW")

Dados carregados: 175,205 linhas
Periodo: 2021-01-01 00:00:00 a 2024-12-31 00:00:00
Regioes: ['Alentejo' 'Algarve' 'Centro' 'Lisboa' 'Norte']
Consumo - Min: 150.00 MW, Max: 4036.51 MW


## 3. Feature Engineering (Sem Lags)

In [3]:
# Criar apenas features temporais e de interacao (SEM lags e rolling windows)
df_features = df.copy()

# Add latitude/longitude based on region (approximate)
region_coords = {
    'Alentejo': (38.5, -7.9),
    'Algarve': (37.1, -8.0),
    'Centro': (40.2, -8.4),
    'Lisboa': (38.7, -9.1),
    'Norte': (41.5, -8.4)
}
df_features['latitude'] = df_features['region'].map(region_coords).apply(lambda x: x[0])
df_features['longitude'] = df_features['region'].map(region_coords).apply(lambda x: x[1])

# Weather derived features
df_features['temperature_feels_like'] = df_features['temperature']  # Simplified

# Temporal features
df_features['hour'] = df_features['timestamp'].dt.hour
df_features['day_of_week'] = df_features['timestamp'].dt.dayofweek
df_features['day_of_month'] = df_features['timestamp'].dt.day
df_features['month'] = df_features['timestamp'].dt.month
df_features['quarter'] = df_features['timestamp'].dt.quarter
df_features['year'] = df_features['timestamp'].dt.year
df_features['week_of_year'] = df_features['timestamp'].dt.isocalendar().week.astype(int)
df_features['day_of_year'] = df_features['timestamp'].dt.dayofyear

# Holiday features (use actual data if available)
df_features['is_holiday'] = df_features['holiday_name'].notna().astype(int)
df_features['is_holiday_eve'] = 0  # Simplified
df_features['is_holiday_after'] = 0  # Simplified
df_features['days_to_holiday'] = 365  # Simplified
df_features['days_from_holiday'] = 365  # Simplified

# Cyclical features (sin_ and cos_ prefix)
df_features['sin_hour'] = np.sin(2 * np.pi * df_features['hour'] / 24)
df_features['cos_hour'] = np.cos(2 * np.pi * df_features['hour'] / 24)
df_features['sin_day_of_week'] = np.sin(2 * np.pi * df_features['day_of_week'] / 7)
df_features['cos_day_of_week'] = np.cos(2 * np.pi * df_features['day_of_week'] / 7)
df_features['sin_month'] = np.sin(2 * np.pi * df_features['month'] / 12)
df_features['cos_month'] = np.cos(2 * np.pi * df_features['month'] / 12)
df_features['sin_day_of_year'] = np.sin(2 * np.pi * df_features['day_of_year'] / 365)
df_features['cos_day_of_year'] = np.cos(2 * np.pi * df_features['day_of_year'] / 365)

# Also keep old names for backward compatibility
df_features['hour_sin'] = df_features['sin_hour']
df_features['hour_cos'] = df_features['cos_hour']
df_features['day_of_week_sin'] = df_features['sin_day_of_week']
df_features['day_of_week_cos'] = df_features['cos_day_of_week']
df_features['month_sin'] = df_features['sin_month']
df_features['month_cos'] = df_features['cos_month']

# Time flags
df_features['is_weekend'] = (df_features['day_of_week'] >= 5).astype(int)
df_features['is_business_hour'] = ((df_features['hour'] >= 9) & (df_features['hour'] < 18) & (df_features['day_of_week'] < 5)).astype(int)

# Periods of day
df_features['is_morning'] = ((df_features['hour'] >= 6) & (df_features['hour'] < 12)).astype(int)
df_features['is_afternoon'] = ((df_features['hour'] >= 12) & (df_features['hour'] < 18)).astype(int)
df_features['is_evening'] = ((df_features['hour'] >= 18) & (df_features['hour'] < 22)).astype(int)
df_features['is_night'] = ((df_features['hour'] >= 22) | (df_features['hour'] < 6)).astype(int)

# Interaction features (weather x temporal)
df_features['temp_hour'] = df_features['temperature'] * df_features['hour']
df_features['temp_weekend'] = df_features['temperature'] * df_features['is_weekend']
df_features['wind_hour'] = df_features['wind_speed'] * df_features['hour']

# One-hot encoding for region (ensure all 5 regions exist)
all_regions = ['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte']
for region in all_regions:
    df_features[f'region_{region}'] = (df_features['region'] == region).astype(int)

# Remover NaN (se houver)
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
df_features = df_features.dropna(subset=numeric_cols)

print(f"\nDados apos feature engineering: {len(df_features):,} linhas")
print(f"Total de features: {df_features.shape[1]} colunas")
print(f"Retencao de dados: {len(df_features)/len(df)*100:.1f}%")


Dados apos feature engineering: 175,205 linhas
Total de features: 56 colunas
Retencao de dados: 100.0%


## 4. Preparar Features e Target

In [4]:
# Excluir colunas nao numericas e target
exclude_cols = ['timestamp', 'consumption_mw', 'region', 'holiday_name']

# Adicionar colunas datetime que podem ter sido criadas
for col in df_features.columns:
    if col in exclude_cols:
        continue
    if pd.api.types.is_datetime64_any_dtype(df_features[col]):
        exclude_cols.append(col)
    if df_features[col].dtype == 'object':
        exclude_cols.append(col)

# Features finais
features = [c for c in df_features.columns if c not in exclude_cols]
target = 'consumption_mw'

print(f"Features selecionadas: {len(features)}")
print(f"Target: {target}")
print(f"\nFeatures disponiveis:")
print(features)

Features selecionadas: 50
Target: consumption_mw

Features disponiveis:
['latitude', 'longitude', 'temperature', 'temperature_feels_like', 'humidity', 'wind_speed', 'precipitation', 'cloud_cover', 'pressure', 'is_holiday', 'is_holiday_eve', 'is_holiday_after', 'days_to_holiday', 'days_from_holiday', 'hour', 'day_of_week', 'day_of_month', 'month', 'quarter', 'day_of_year', 'week_of_year', 'is_weekend', 'is_business_hour', 'sin_hour', 'cos_hour', 'sin_day_of_week', 'cos_day_of_week', 'sin_month', 'cos_month', 'sin_day_of_year', 'cos_day_of_year', 'year', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'is_morning', 'is_afternoon', 'is_evening', 'is_night', 'temp_hour', 'temp_weekend', 'wind_hour', 'region_Alentejo', 'region_Algarve', 'region_Centro', 'region_Lisboa', 'region_Norte']


## 5. Split Temporal (70% Train / 15% Val / 15% Test)

In [5]:
# Ordenar por timestamp
df_features = df_features.sort_values('timestamp').reset_index(drop=True)

# Split temporal 70/15/15
n = len(df_features)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train = df_features.iloc[:train_end]
val = df_features.iloc[train_end:val_end]
test = df_features.iloc[val_end:]

# Extrair X e y
X_train = train[features].values
y_train = train[target].values

X_val = val[features].values
y_val = val[target].values

X_test = test[features].values
y_test = test[target].values

# Guardar timestamps e regioes para analise posterior
timestamps_test = test['timestamp'].reset_index(drop=True)
regions_test = test['region'].reset_index(drop=True)

print("="*70)
print("SPLIT TEMPORAL")
print("="*70)
print(f"Train: {len(X_train):,} amostras ({len(X_train)/n*100:.1f}%)")
print(f"Val:   {len(X_val):,} amostras ({len(X_val)/n*100:.1f}%)")
print(f"Test:  {len(X_test):,} amostras ({len(X_test)/n*100:.1f}%)")
print("="*70)
print(f"\nPeriodo Train: {train['timestamp'].min()} a {train['timestamp'].max()}")
print(f"Periodo Val:   {val['timestamp'].min()} a {val['timestamp'].max()}")
print(f"Periodo Test:  {test['timestamp'].min()} a {test['timestamp'].max()}")

SPLIT TEMPORAL
Train: 122,643 amostras (70.0%)
Val:   26,281 amostras (15.0%)
Test:  26,281 amostras (15.0%)

Periodo Train: 2021-01-01 00:00:00 a 2023-10-20 00:00:00
Periodo Val:   2023-10-20 00:00:00 a 2024-05-26 00:00:00
Periodo Test:  2024-05-26 00:00:00 a 2024-12-31 00:00:00


## 6. Treinar Modelos

In [6]:
# Inicializar avaliador
evaluator = ModelEvaluator(output_dir='../data/models')

# Dicionarios para armazenar modelos e resultados
models = {}
results = []

print("\nIniciando treinamento dos modelos SEM LAGS...")
print("="*70)


Iniciando treinamento dos modelos SEM LAGS...


### 6.1 Random Forest

In [7]:
print("\n[1/4] Treinando Random Forest...")

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf.fit(X_train, y_train)

# Validacao
y_val_pred = rf.predict(X_val)
metrics = evaluator.calculate_metrics(y_val, y_val_pred, prefix='val_')

print(f"  Val - MAE: {metrics['val_mae']:.2f} MW | RMSE: {metrics['val_rmse']:.2f} MW | R2: {metrics['val_r2']:.4f}")

models['random_forest'] = rf
results.append({'model': 'Random Forest', **metrics})


[1/4] Treinando Random Forest...
  Val - MAE: 58.68 MW | RMSE: 86.36 MW | R2: 0.9895


### 6.2 XGBoost

In [8]:
print("\n[2/4] Treinando XGBoost...")

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

# Validacao
y_val_pred = xgb_model.predict(X_val)
metrics = evaluator.calculate_metrics(y_val, y_val_pred, prefix='val_')

print(f"  Val - MAE: {metrics['val_mae']:.2f} MW | RMSE: {metrics['val_rmse']:.2f} MW | R2: {metrics['val_r2']:.4f}")

models['xgboost'] = xgb_model
results.append({'model': 'XGBoost', **metrics})


[2/4] Treinando XGBoost...
  Val - MAE: 57.08 MW | RMSE: 83.67 MW | R2: 0.9901


### 6.3 LightGBM

In [9]:
print("\n[3/4] Treinando LightGBM...")

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.05,
    num_leaves=50,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)]
)

# Validacao
y_val_pred = lgb_model.predict(X_val)
metrics = evaluator.calculate_metrics(y_val, y_val_pred, prefix='val_')

print(f"  Val - MAE: {metrics['val_mae']:.2f} MW | RMSE: {metrics['val_rmse']:.2f} MW | R2: {metrics['val_r2']:.4f}")

models['lightgbm'] = lgb_model
results.append({'model': 'LightGBM', **metrics})


[3/4] Treinando LightGBM...
  Val - MAE: 55.31 MW | RMSE: 80.41 MW | R2: 0.9909


### 6.4 CatBoost

In [10]:
print("\n[4/4] Treinando CatBoost...")

cat_model = CatBoostRegressor(
    iterations=500,
    depth=10,
    learning_rate=0.05,
    l2_leaf_reg=3,
    random_state=42,
    verbose=False
)
cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    verbose=False
)

# Validacao
y_val_pred = cat_model.predict(X_val)
metrics = evaluator.calculate_metrics(y_val, y_val_pred, prefix='val_')

print(f"  Val - MAE: {metrics['val_mae']:.2f} MW | RMSE: {metrics['val_rmse']:.2f} MW | R2: {metrics['val_r2']:.4f}")

models['catboost'] = cat_model
results.append({'model': 'CatBoost', **metrics})

print("\n" + "="*70)
print("Todos os modelos treinados!")
print("="*70)


[4/4] Treinando CatBoost...
  Val - MAE: 54.98 MW | RMSE: 80.02 MW | R2: 0.9910

Todos os modelos treinados!


## 7. Comparar Modelos

In [11]:
# Criar DataFrame com resultados
df_results = pd.DataFrame(results).sort_values('val_rmse')

print("\n" + "="*90)
print("COMPARACAO DE MODELOS (SEM LAGS) - VALIDACAO")
print("="*90)
print(df_results[['model', 'val_mae', 'val_rmse', 'val_mape', 'val_r2']].to_string(index=False))
print("="*90)

# Identificar melhor modelo
best_name = df_results.iloc[0]['model']
best_metrics = df_results.iloc[0]

print(f"\n🏆 MELHOR MODELO (SEM LAGS): {best_name}")
print(f"   MAE:  {best_metrics['val_mae']:.2f} MW")
print(f"   RMSE: {best_metrics['val_rmse']:.2f} MW")
print(f"   MAPE: {best_metrics['val_mape']:.2f}%")
print(f"   R²:   {best_metrics['val_r2']:.4f}")


COMPARACAO DE MODELOS (SEM LAGS) - VALIDACAO
        model   val_mae  val_rmse  val_mape   val_r2
     CatBoost 54.976428 80.020733  4.485043 0.990957
     LightGBM 55.307421 80.405422  4.512152 0.990870
      XGBoost 57.077732 83.671011  4.657867 0.990113
Random Forest 58.675100 86.356798  4.764434 0.989468

🏆 MELHOR MODELO (SEM LAGS): CatBoost
   MAE:  54.98 MW
   RMSE: 80.02 MW
   MAPE: 4.49%
   R²:   0.9910


## 8. Avaliar no Conjunto de Teste

In [12]:
# Selecionar melhor modelo
best_key = best_name.lower().replace(' ', '_')
best_model = models[best_key]

print(f"\nAvaliando {best_name} no conjunto de teste...")

# Predicao no teste
y_test_pred = best_model.predict(X_test)

# Calcular metricas de teste
test_metrics = evaluator.calculate_metrics(y_test, y_test_pred, prefix='test_')

print("\n" + "="*70)
print(f"RESULTADOS NO TESTE - {best_name} (SEM LAGS)")
print("="*70)
print(f"MAE:   {test_metrics['test_mae']:.2f} MW")
print(f"RMSE:  {test_metrics['test_rmse']:.2f} MW")
print(f"MAPE:  {test_metrics['test_mape']:.2f}%")
print(f"R²:    {test_metrics['test_r2']:.4f}")
print(f"NRMSE: {test_metrics['test_nrmse']:.4f}")
print("="*70)


Avaliando CatBoost no conjunto de teste...

RESULTADOS NO TESTE - CatBoost (SEM LAGS)
MAE:   58.19 MW
RMSE:  86.55 MW
MAPE:  4.54%
R²:    0.9900
NRMSE: 0.0634


## 9. Salvar Modelo Sem Lags

In [13]:
# Criar diretorio de output
output_dir = Path('../data/models')
output_dir.mkdir(parents=True, exist_ok=True)

print("\nSalvando modelo SEM LAGS...")
print("="*70)

# Salvar melhor modelo com sufixo _no_lags
best_model_path = output_dir / f'{best_key}_no_lags.pkl'
joblib.dump(best_model, best_model_path)
print(f"✓ Melhor modelo (sem lags): {best_key}_no_lags.pkl")

# Salvar feature names com sufixo _no_lags
with open(output_dir / 'feature_names_no_lags.txt', 'w') as f:
    for feature in features:
        f.write(f"{feature}\n")
print(f"✓ feature_names_no_lags.txt ({len(features)} features)")

# Salvar comparacao de modelos
df_results.to_csv(output_dir / 'model_comparison_no_lags.csv', index=False)
print(f"✓ model_comparison_no_lags.csv")

# Salvar metadata do treinamento
metadata = {
    'model_type': 'no_lags',
    'best_model': best_name,
    'training_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'features_used': 'temporal + weather + interactions (NO LAGS)',
    'data_info': {
        'total_samples': int(len(df_features)),
        'train_size': int(len(X_train)),
        'val_size': int(len(X_val)),
        'test_size': int(len(X_test)),
        'n_features': int(len(features)),
        'train_period': f"{train['timestamp'].min()} to {train['timestamp'].max()}",
        'test_period': f"{test['timestamp'].min()} to {test['timestamp'].max()}"
    },
    'test_metrics': {
        'mae': float(test_metrics['test_mae']),
        'rmse': float(test_metrics['test_rmse']),
        'mape': float(test_metrics['test_mape']),
        'r2': float(test_metrics['test_r2']),
        'nrmse': float(test_metrics['test_nrmse'])
    },
    'model_comparison': df_results.to_dict('records'),
    'note': 'This model does NOT use lag features and can make predictions without historical consumption data'
}

with open(output_dir / 'training_metadata_no_lags.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ training_metadata_no_lags.json")

print("="*70)
print("Todos os arquivos salvos em: data/models/")
print("Sufixo: _no_lags")


Salvando modelo SEM LAGS...
✓ Melhor modelo (sem lags): catboost_no_lags.pkl
✓ feature_names_no_lags.txt (50 features)
✓ model_comparison_no_lags.csv
✓ training_metadata_no_lags.json
Todos os arquivos salvos em: data/models/
Sufixo: _no_lags


## 10. Resumo Final

In [14]:
print("\n" + "="*90)
print("RESUMO DO TREINAMENTO (SEM LAGS)")
print("="*90)

print(f"\n📊 DADOS")
print(f"   Total de amostras: {len(df_features):,}")
print(f"   Features: {len(features)} (SEM lags ou rolling windows)")
print(f"   Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

print(f"\n🤖 MODELOS TREINADOS: {len(models)}")
for model_name in df_results['model'].values:
    print(f"   - {model_name}")

print(f"\n🏆 MELHOR MODELO: {best_name}")
print(f"   Validacao - RMSE: {best_metrics['val_rmse']:.2f} MW | MAPE: {best_metrics['val_mape']:.2f}% | R²: {best_metrics['val_r2']:.4f}")
print(f"   Teste     - RMSE: {test_metrics['test_rmse']:.2f} MW | MAPE: {test_metrics['test_mape']:.2f}% | R²: {test_metrics['test_r2']:.4f}")

print(f"\n✅ VANTAGENS")
print(f"   - Funciona SEM historico de consumo")
print(f"   - Usa apenas: weather + temporal features")
print(f"   - Ideal para API sem base de dados historica")

print(f"\n⚠️  TRADE-OFF")
print(f"   - Menor precisao que modelo com lags")
print(f"   - Esperado: MAPE ~3-8% (vs 0.86% com lags)")

print(f"\n📁 OUTPUTS SALVOS")
print(f"   - Modelo: {best_key}_no_lags.pkl")
print(f"   - Features: feature_names_no_lags.txt")
print(f"   - Metadados: training_metadata_no_lags.json")

print("\n" + "="*90)
print(f"Fim do treinamento: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*90)


RESUMO DO TREINAMENTO (SEM LAGS)

📊 DADOS
   Total de amostras: 175,205
   Features: 50 (SEM lags ou rolling windows)
   Train: 122,643 | Val: 26,281 | Test: 26,281

🤖 MODELOS TREINADOS: 4
   - CatBoost
   - LightGBM
   - XGBoost
   - Random Forest

🏆 MELHOR MODELO: CatBoost
   Validacao - RMSE: 80.02 MW | MAPE: 4.49% | R²: 0.9910
   Teste     - RMSE: 86.55 MW | MAPE: 4.54% | R²: 0.9900

✅ VANTAGENS
   - Funciona SEM historico de consumo
   - Usa apenas: weather + temporal features
   - Ideal para API sem base de dados historica

⚠️  TRADE-OFF
   - Menor precisao que modelo com lags
   - Esperado: MAPE ~3-8% (vs 0.86% com lags)

📁 OUTPUTS SALVOS
   - Modelo: catboost_no_lags.pkl
   - Features: feature_names_no_lags.txt
   - Metadados: training_metadata_no_lags.json

Fim do treinamento: 2025-10-07 10:39:41
